In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import requests
import io
import zipfile
import re

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 24
PROJECT_ROOT: /Users/boulanger/Documents/governance-framework


## IMF iMaPP Pipeline

**Source:** IMF Integrated Macroprudential Policy Database
**Access:** Automated direct ZIP download — no registration required
**Download instructions:** See `docs/instructions_data_maintenance.md` — IMF_IMAPP section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| Macroprudential policy tightening/loosening actions | Macroeconomic policy framework quality | Primary tier 1 |

In [2]:
import requests
import zipfile
import io
import re
from datetime import datetime, timedelta
import pandas as pd

IMAPP_BASE = "https://www.elibrary-areaer.imf.org/Macroprudential/Documents"

def get_latest_imapp_url():
    """Auto-detect latest iMaPP ZIP by iterating dates backwards from today."""
    check_date = datetime.today()
    for _ in range(730):  # Search up to 2 years back
        date_str = check_date.strftime("%Y-%m-%d")
        url = f"{IMAPP_BASE}/iMaPP_database-{date_str}.zip"
        try:
            r = requests.head(url, timeout=5, allow_redirects=True)
            if r.status_code == 200 and 'zip' in r.headers.get('Content-Type', '').lower():
                print(f"Found latest iMaPP: {date_str}")
                return url, date_str
        except:
            pass
        check_date -= timedelta(days=1)
    return None, None

IMAPP_URL, IMAPP_DATE = get_latest_imapp_url()

if IMAPP_URL:
    print(f"\nDownloading iMaPP {IMAPP_DATE}...")
    response = requests.get(IMAPP_URL, timeout=120)
    print(f"Status: {response.status_code}, Size: {len(response.content)/1024/1024:.1f}MB")
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        print(f"ZIP contents: {z.namelist()}")

Found latest iMaPP: 2025-09-29

Status: 200, Size: 7.9MB
ZIP contents: ['Alam et al. (2019) iMaPP WP.pdf', 'iMaPP_database-2025-9-29.xlsx', 'iMaPP_load.do', 'ReadMe_Main.txt', 'Sample Files for Figures Alam et al. (2019)/', 'Sample Files for Figures Alam et al. (2019)/iMaPP_Fig123.do', 'Sample Files for Figures Alam et al. (2019)/iMaPP_Fig1234.xlsx', 'Sample Files for Figures Alam et al. (2019)/iMaPP_Fig4.do', 'Sample Files for Figures Alam et al. (2019)/iMaPP_Fig5.do', 'Sample Files for Figures Alam et al. (2019)/iMaPP_Fig5a.emf', 'Sample Files for Figures Alam et al. (2019)/iMaPP_Fig5b.emf', 'Sample Files for Figures Alam et al. (2019)/iMaPP_LTV_average_statistics.docx', 'Sample Files for Figures Alam et al. (2019)/ReadMe_SampleFiles.txt', 'Sample Files for Figures Alam et al. (2019)/Thumbs.db']


In [3]:
# Inspect MaPP sheet structure
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    excel_file = [f for f in z.namelist() if f.endswith('.xlsx') and 'iMaPP_database' in f][0]
    xl = pd.ExcelFile(io.BytesIO(z.read(excel_file)), engine='openpyxl')
    
    mapp = xl.parse('MaPP')
    ltv = xl.parse('LTV_average')
    ccb = xl.parse('CCB')

print("MaPP sheet:")
print(f"  Shape: {mapp.shape}")
print(f"  Columns: {list(mapp.columns[:10])}")
print(mapp.head(3))

print("\nLTV_average sheet:")
print(f"  Shape: {ltv.shape}")
print(f"  Columns: {list(ltv.columns[:10])}")
print(ltv.head(3))

MaPP sheet:
  Shape: (56700, 36)
  Columns: ['Country', 'ifscode', 'iso3', 'iso2', 'AE', 'EMDE', 'Year', 'Month', 'CCB', 'Conservation']
   Country  ifscode iso3 iso2   AE  EMDE  Year  Month  CCB  Conservation  ...  \
0  Albania      914  ALB   AL  0.0   1.0  1990      1  0.0           0.0  ...   
1  Albania      914  ALB   AL  0.0   1.0  1990      2  0.0           0.0  ...   
2  Albania      914  ALB   AL  0.0   1.0  1990      3  0.0           0.0  ...   

   DSTI  Tax  Liquidity  LTD  LFX  RR  RR_FCD  SIFI   OT  SUM_17  
0   0.0  0.0          0  0.0  0.0   0       0   0.0  0.0       0  
1   0.0  0.0          0  0.0  0.0   0       0   0.0  0.0       0  
2   0.0  0.0          0  0.0  0.0   0       0   0.0  0.0       0  

[3 rows x 36 columns]

LTV_average sheet:
  Shape: (27720, 10)
  Columns: ['Country', 'ifscode', 'iso3', 'iso2', 'AE', 'EMDE', 'Year', 'Month', 'LTV_average', 'LTV_median']
     Country  ifscode iso3 iso2  AE  EMDE  Year  Month  LTV_average  \
0  Argentina      213  AR

In [5]:

print("MaPP_T columns:", list(mapp_t.columns))
print("MaPP_L columns:", list(mapp_l.columns))
print("\nMaPP_T head:")
print(mapp_t.head(3))

MaPP_T columns: ['Country', 'ifscode', 'iso3', 'iso2', 'AE', 'EMDE', 'Year', 'Month', 'CCB_T', 'Conservation_T', 'Capital_T', 'Capital_Gen_T', 'Capital_HH_T', 'Capital_Corp_T', 'Capital_FX_T', 'LVR_T', 'LLP_T', 'LCG_T', 'LCG_Gen_T', 'LCG_HH_T', 'LCG_Corp_T', 'LoanR_T', 'LoanR_HH_T', 'LoanR_Corp_T', 'LFC_T', 'LTV_T', 'DSTI_T', 'Tax_T', 'Liquidity_T', 'LTD_T', 'LFX_T', 'RR_T', 'RR_FCD_T', 'SIFI_T', 'OT_T', 'SUM_17_T']
MaPP_L columns: ['Country', 'ifscode', 'iso3', 'iso2', 'AE', 'EMDE', 'Year', 'Month', 'CCB_L', 'Conservation_L', 'Capital_L', 'Capital_Gen_L', 'Capital_HH_L', 'Capital_Corp_L', 'Capital_FX_L', 'LVR_L', 'LLP_L', 'LCG_L', 'LCG_Gen_L', 'LCG_HH_L', 'LCG_Corp_L', 'LoanR_L', 'LoanR_HH_L', 'LoanR_Corp_L', 'LFC_L', 'LTV_L', 'DSTI_L', 'Tax_L', 'Liquidity_L', 'LTD_L', 'LFX_L', 'RR_L', 'RR_FCD_L', 'SIFI_L', 'OT_L', 'SUM_17_L']

MaPP_T head:
   Country  ifscode iso3 iso2   AE  EMDE  Year  Month  CCB_T  Conservation_T  \
0  Albania      914  ALB   AL  0.0   1.0  1990      1    0.0      

In [6]:
# Aggregate monthly data to country-year
# Sum tightening and loosening actions per country per year
# Take annual average of LTV limit

# Tightening actions — annual sum
t_cy = mapp_t.groupby(['iso3', 'Country', 'Year'])['SUM_17_T'].sum().reset_index()
t_cy = t_cy.rename(columns={'SUM_17_T': 'imapp_tightening_actions'})

# Loosening actions — annual sum
l_cy = mapp_l.groupby(['iso3', 'Country', 'Year'])['SUM_17_L'].sum().reset_index()
l_cy = l_cy.rename(columns={'SUM_17_L': 'imapp_loosening_actions'})

# LTV average — annual mean
ltv_cy = ltv.groupby(['iso3', 'Country', 'Year'])['LTV_average'].mean().reset_index()
ltv_cy = ltv_cy.rename(columns={'LTV_average': 'imapp_ltv_average'})

# Merge all
imapp = t_cy.merge(l_cy, on=['iso3', 'Country', 'Year'], how='outer')
imapp = imapp.merge(ltv_cy, on=['iso3', 'Country', 'Year'], how='outer')

# Rename and standardise
imapp = imapp.rename(columns={
    'iso3':    'country_code',
    'Country': 'country_name',
    'Year':    'year',
})

# Derive net tightening (tightening minus loosening)
imapp['imapp_net_tightening'] = imapp['imapp_tightening_actions'] - imapp['imapp_loosening_actions']

# Filter to framework start year
imapp = imapp[imapp['year'] >= FRAMEWORK_START_YEAR].copy()
imapp = imapp.sort_values(['country_name', 'year']).reset_index(drop=True)

print(f"Shape: {imapp.shape}")
print(f"Years: {imapp['year'].min()} — {imapp['year'].max()}")
print(f"Countries: {imapp['country_name'].nunique()}")
print(f"\nMissing values (%):")
missing_pct = (imapp.isnull().sum() / len(imapp) * 100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))
print(imapp.head(3))

Shape: (4725, 7)
Years: 1990 — 2024
Countries: 135

Missing values (%):
imapp_ltv_average    52.0
dtype: float64
  country_code country_name  year  imapp_tightening_actions  \
0          ALB      Albania  1990                         0   
1          ALB      Albania  1991                         0   
2          ALB      Albania  1992                         0   

   imapp_loosening_actions  imapp_ltv_average  imapp_net_tightening  
0                        0                NaN                     0  
1                        0                NaN                     0  
2                        0                NaN                     0  


In [7]:
# Derive data currency from ZIP filename — no hardcoding
data_as_of_date = f"{IMAPP_DATE[:4]}-{IMAPP_DATE[5:7]}"

# Save to processed
output_path = os.path.join(PROCESSED_DIR, "imapp_clean.csv")
imapp.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {imapp.shape}")

# Update download log
update_entry(
    "IMF_IMAPP",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=data_as_of_date,
    local_filename="imapp_clean.csv",
    latest_available_version=IMAPP_DATE,
    notes=f"Macroprudential policy tightening/loosening actions and LTV limits. Aggregated from monthly to annual. Auto-detects latest ZIP by date iteration. Coverage: 1990-2024, 135 countries."
)

print_entry("IMF_IMAPP")

Written: /Users/boulanger/Documents/governance-framework/data/processed/imapp_clean.csv
Shape: (4725, 7)
[download_log] Updated entry for IMF_IMAPP
  source_id: IMF_IMAPP
  last_attempted_date: 2026-06-12
  last_successful_download_date: 2026-06-12
  data_as_of_date: 2025-09
  local_filename: imapp_clean.csv
  latest_available_version: 2025-09-29
  no_update_reason: nan
  notes: Macroprudential policy tightening/loosening actions and LTV limits. Aggregated from monthly to annual. Auto-detects latest ZIP by date iteration. Coverage: 1990-2024, 135 countries.
